In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp
from pyspark.sql import Window

In [0]:

CATALOG = spark.conf.get("catalog")
SILVER_SCHEMA = spark.conf.get("silver_schema")
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_view_events"

GOLD_SCHEMA = spark.conf.get("gold_schema")

DIM_SHOW = f"{CATALOG}.{GOLD_SCHEMA}.dim_show"
DIM_DATE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
DIM_USER = f"{CATALOG}.{GOLD_SCHEMA}.dim_user"

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.fact_views")
def fact_views():
    silver_df = spark.read.table(SILVER_TABLE)

    dim_show_df = spark.read.table(DIM_SHOW)
    dim_date_df = spark.read.table(DIM_DATE)
    dim_user_df = spark.read.table(DIM_USER)

    return (
        silver_df
        .join(
            dim_show_df.select("show_id", "show_key"),
            on="show_id",
            how="left"
        )
        .join(
            dim_date_df.select("view_date", "date_key"),
            on="view_date",
            how="left"
        )
        .join(
            dim_user_df.select("user_id", "user_key"),
            on="user_id",
            how="left"
        )
        .select(
            "event_id",
            "show_key",
            "date_key",
            "user_key",
            "rate"
        )
    )